<a href="https://colab.research.google.com/github/vaibhavjiyer87/engine-nvh-deep-learning/blob/main/notebooks/10_multitask_rpm_torque_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 1 — Mount Google Drive

# ============================================================
# CELL 1 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Persistent project location in Google Drive
PROJECT_DRIVE = Path(
    "/content/drive/MyDrive/"
    "NVH_DeepLearning/01_EngineOperatingState"
)

if not PROJECT_DRIVE.exists():
    raise FileNotFoundError(
        f"Project folder was not found:\n{PROJECT_DRIVE}"
    )

print("Google Drive mounted successfully.")
print(f"Project Drive: {PROJECT_DRIVE}")

Mounted at /content/drive
Google Drive mounted successfully.
Project Drive: /content/drive/MyDrive/NVH_DeepLearning/01_EngineOperatingState


In [2]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 2 — Authenticate GitHub + determine Git author identity
# This assumes your GitHub token is stored in Colab Secrets under:
# GITHUB_TOKEN

# ============================================================
# CELL 2 — AUTHENTICATE GITHUB
# ============================================================

from google.colab import userdata

import os
import shutil
import subprocess


# ------------------------------------------------------------
# Repository settings
# ------------------------------------------------------------

REPOSITORY_NAME = "engine-nvh-deep-learning"


# ------------------------------------------------------------
# Load GitHub token securely from Colab Secrets
# ------------------------------------------------------------

github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError(
        "GITHUB_TOKEN was not found in Colab Secrets.\n"
        "Add the token and enable Notebook access."
    )

# GitHub CLI recognizes GH_TOKEN automatically.
os.environ["GH_TOKEN"] = github_token
os.environ["GH_HOST"] = "github.com"


# ------------------------------------------------------------
# Install GitHub CLI if necessary
# ------------------------------------------------------------

if shutil.which("gh") is None:

    print("Installing GitHub CLI...")

    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True,
    )

    subprocess.run(
        [
            "apt-get",
            "install",
            "-y",
            "-qq",
            "gh",
        ],
        check=True,
    )


# ------------------------------------------------------------
# Verify authentication
# ------------------------------------------------------------

auth_result = subprocess.run(
    [
        "gh",
        "api",
        "user",
        "--jq",
        ".login",
    ],
    capture_output=True,
    text=True,
)

if auth_result.returncode != 0:

    print(auth_result.stderr)

    raise RuntimeError(
        "GitHub authentication failed."
    )


GITHUB_USERNAME = auth_result.stdout.strip()

# ------------------------------------------------------------
# Configure Git commit identity
# ------------------------------------------------------------

user_id_result = subprocess.run(
    [
        "gh",
        "api",
        "user",
        "--jq",
        ".id",
    ],
    capture_output=True,
    text=True,
    check=True,
)

GITHUB_USER_ID = (
    user_id_result.stdout.strip()
)

GIT_NAME = GITHUB_USERNAME

GIT_EMAIL = (
    f"{GITHUB_USER_ID}+"
    f"{GITHUB_USERNAME}@users.noreply.github.com"
)

print("Git identity prepared.")
print(f"Name:  {GIT_NAME}")
print(f"Email: {GIT_EMAIL}")

# ------------------------------------------------------------
# Configure Git to use GitHub CLI authentication
# ------------------------------------------------------------

subprocess.run(
    [
        "gh",
        "auth",
        "setup-git",
        "--hostname",
        "github.com",
        "--force",
    ],
    check=True,
)


print("GitHub authentication successful.")
print(f"GitHub user: {GITHUB_USERNAME}")
print(f"Repository:  {REPOSITORY_NAME}")

Git identity prepared.
Name:  vaibhavjiyer87
Email: 312107027+vaibhavjiyer87@users.noreply.github.com
GitHub authentication successful.
GitHub user: vaibhavjiyer87
Repository:  engine-nvh-deep-learning


In [3]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 3 — Clone repository if .git is missing / Update repository
# This cell deals with the temporary nature of /content.

# ============================================================
# CELL 3 — RESTORE LOCAL GITHUB REPOSITORY
# ============================================================

from pathlib import Path
import shutil
import subprocess


TEMP_REPO_DIR = (
    Path("/content")
    / REPOSITORY_NAME
)

REPOSITORY_IDENTIFIER = (
    f"{GITHUB_USERNAME}/"
    f"{REPOSITORY_NAME}"
)


# ------------------------------------------------------------
# Case 1:
# Valid Git repository already exists
# ------------------------------------------------------------

if (
    TEMP_REPO_DIR.exists()
    and
    (TEMP_REPO_DIR / ".git").exists()
):

    print(
        "Git repository already exists "
        "in this Colab runtime."
    )

    # Pull updates only when the working tree is clean.
    status_result = subprocess.run(
        [
            "git",
            "-C",
            str(TEMP_REPO_DIR),
            "status",
            "--porcelain",
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    if status_result.stdout.strip():

        print(
            "Local changes detected."
        )

        print(
            "Automatic git pull skipped "
            "to avoid overwriting local work."
        )

    else:

        print(
            "Working tree is clean. "
            "Updating from GitHub..."
        )

        pull_result = subprocess.run(
            [
                "git",
                "-C",
                str(TEMP_REPO_DIR),
                "pull",
                "--ff-only",
            ],
            capture_output=True,
            text=True,
        )

        print(pull_result.stdout)

        if pull_result.returncode != 0:
            print(pull_result.stderr)


# ------------------------------------------------------------
# Case 2:
# Folder exists, but it is NOT a Git repository
# ------------------------------------------------------------

elif TEMP_REPO_DIR.exists():

    raise RuntimeError(
        f"The folder exists but is not a Git repository:\n"
        f"{TEMP_REPO_DIR}\n\n"
        "Do not run git init. Inspect or back up the folder "
        "before removing it and rerunning this cell."
    )


# ------------------------------------------------------------
# Case 3:
# Fresh runtime — clone repository
# ------------------------------------------------------------

else:

    print(
        "Repository not present in this runtime."
    )

    print(
        f"Cloning {REPOSITORY_IDENTIFIER}..."
    )

    clone_result = subprocess.run(
        [
            "gh",
            "repo",
            "clone",
            REPOSITORY_IDENTIFIER,
            str(TEMP_REPO_DIR),
        ],
        capture_output=True,
        text=True,
    )

    print(clone_result.stdout)

    if clone_result.returncode != 0:

        print(clone_result.stderr)

        raise RuntimeError(
            "Repository clone failed."
        )


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

if not (
    TEMP_REPO_DIR
    / ".git"
).exists():

    raise RuntimeError(
        "Repository restoration failed."
    )


print("Local Git repository is ready.")
print(f"Location: {TEMP_REPO_DIR}")

Repository not present in this runtime.
Cloning vaibhavjiyer87/engine-nvh-deep-learning...

Local Git repository is ready.
Location: /content/engine-nvh-deep-learning


In [4]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 4 — Define REPO_DIR and all standard paths + restore requirements

# ============================================================
# CELL 4 — DEFINE PROJECT PATHS
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# GitHub working repository
# ------------------------------------------------------------

REPO_DIR = (
    Path("/content")
    / REPOSITORY_NAME
)

if not (
    REPO_DIR
    / ".git"
).exists():

    raise FileNotFoundError(
        f"Valid Git repository not found at:\n"
        f"{REPO_DIR}\n\n"
        "Run Cell 3 first."
    )


# ------------------------------------------------------------
# Persistent raw dataset
# ------------------------------------------------------------

RAW_ROOT = (
    PROJECT_DRIVE
    / "data"
    / "raw"
    / "procedural_engine_sounds"
)

DATASET_ROOT = (
    RAW_ROOT
    / "dataset"
)

AUDIO_DIR = (
    DATASET_ROOT
    / "audio"
    / "A_full_set"
)


# ------------------------------------------------------------
# Persistent Google Drive manifests
# ------------------------------------------------------------

DRIVE_MANIFEST_DIR = (
    PROJECT_DRIVE
    / "data"
    / "manifests"
)


# ------------------------------------------------------------
# GitHub configuration
# ------------------------------------------------------------

CONFIG_DIR = (
    REPO_DIR
    / "configs"
)

REPORT_DIR = (
    REPO_DIR
    / "reports"
)

SPLIT_DIR = (
    REPO_DIR
    / "data"
    / "splits"
)


# ------------------------------------------------------------
# GitHub result directories
# ------------------------------------------------------------

RESULTS_DIR = (
    REPO_DIR
    / "results"
)

FIGURE_DIR = (
    RESULTS_DIR
    / "figures"
)

TABLE_DIR = (
    RESULTS_DIR
    / "tables"
)

DATA_AUDIT_FIGURE_DIR = (
    FIGURE_DIR
    / "data_audit"
)

DESIGN_FIGURE_DIR = (
    FIGURE_DIR
    / "preprocessing_design"
)

SPLIT_FIGURE_DIR = (
    FIGURE_DIR
    / "split_design"
)


# ------------------------------------------------------------
# Persistent Drive output directories
# ------------------------------------------------------------

DRIVE_OUTPUT_DIR = (
    PROJECT_DRIVE
    / "outputs"
)

DRIVE_DESIGN_TABLE_DIR = (
    DRIVE_OUTPUT_DIR
    / "tables"
    / "preprocessing_design"
)


# ------------------------------------------------------------
# Create output folders if missing
# ------------------------------------------------------------

directories_to_create = [
    CONFIG_DIR,
    REPORT_DIR,
    SPLIT_DIR,
    FIGURE_DIR,
    TABLE_DIR,
    DATA_AUDIT_FIGURE_DIR,
    DESIGN_FIGURE_DIR,
    SPLIT_FIGURE_DIR,
    DRIVE_DESIGN_TABLE_DIR,
]

for directory in directories_to_create:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print("Project paths restored.")
print()
print(f"REPO_DIR:     {REPO_DIR}")
print(f"PROJECT_DRIVE:{PROJECT_DRIVE}")
print(f"RAW_ROOT:     {RAW_ROOT}")
print(f"AUDIO_DIR:    {AUDIO_DIR}")

# ------------------------------------------------------------
# Apply Git commit identity to cloned repository
# ------------------------------------------------------------

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "config",
        "user.name",
        GIT_NAME,
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "config",
        "user.email",
        GIT_EMAIL,
    ],
    check=True,
)

print("Git commit identity applied to repository.")

# ------------------------------------------------------------
# Restore project Python dependencies
# ------------------------------------------------------------

import subprocess

REQUIREMENTS_PATH = (
    REPO_DIR
    / "requirements.txt"
)

if not REQUIREMENTS_PATH.exists():
    raise FileNotFoundError(
        f"requirements.txt not found:\n"
        f"{REQUIREMENTS_PATH}"
    )

install_result = subprocess.run(
    [
        "python",
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REQUIREMENTS_PATH),
    ],
    capture_output=True,
    text=True,
)

if install_result.returncode != 0:

    print(install_result.stdout)
    print(install_result.stderr)

    raise RuntimeError(
        "Project dependency installation failed."
    )

print(
    "Project Python dependencies restored."
)

Project paths restored.

REPO_DIR:     /content/engine-nvh-deep-learning
PROJECT_DRIVE:/content/drive/MyDrive/NVH_DeepLearning/01_EngineOperatingState
RAW_ROOT:     /content/drive/MyDrive/NVH_DeepLearning/01_EngineOperatingState/data/raw/procedural_engine_sounds
AUDIO_DIR:    /content/drive/MyDrive/NVH_DeepLearning/01_EngineOperatingState/data/raw/procedural_engine_sounds/dataset/audio/A_full_set
Git commit identity applied to repository.
Project Python dependencies restored.


In [5]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 5 — Load persistent analysis data
# This cell restores the major tables you've created so far.
# This is conditional, as some files may not exist yet depending on
# where you are in the project.

# ============================================================
# CELL 5 — LOAD PERSISTENT ANALYSIS DATA
# ============================================================

# ============================================================
# STANDARD PROJECT IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import yaml

from tqdm.auto import tqdm

print(
    "Standard project libraries imported."
)


# ------------------------------------------------------------
# 1. Raw-file manifest — REQUIRED
# ------------------------------------------------------------

RAW_MANIFEST_PATH = (
    DRIVE_MANIFEST_DIR
    / "raw_file_manifest_v001.csv"
)

if not RAW_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        f"Required raw manifest not found:\n"
        f"{RAW_MANIFEST_PATH}"
    )


raw_manifest = pd.read_csv(
    RAW_MANIFEST_PATH
)

print(
    f"LOADED raw_manifest: "
    f"{raw_manifest.shape}"
)


# ------------------------------------------------------------
# 2. PREP-001 detailed target analysis — OPTIONAL
# ------------------------------------------------------------

PREP001_TARGET_PATH = (
    DRIVE_DESIGN_TABLE_DIR
    / "prep001_target_analysis_v001.csv.gz"
)

if PREP001_TARGET_PATH.exists():

    prep001_targets = pd.read_csv(
        PREP001_TARGET_PATH
    )

    print(
        f"LOADED prep001_targets: "
        f"{prep001_targets.shape}"
    )

else:

    prep001_targets = None

    print(
        "NOT FOUND: prep001 target analysis "
        "(this is okay if it has not been generated yet)."
    )


# ------------------------------------------------------------
# 3. Window candidate metrics — OPTIONAL
# ------------------------------------------------------------

WINDOW_CANDIDATE_PATH = (
    DRIVE_DESIGN_TABLE_DIR
    / "window_candidate_metrics_v001.csv.gz"
)

if WINDOW_CANDIDATE_PATH.exists():

    window_candidates = pd.read_csv(
        WINDOW_CANDIDATE_PATH
    )

    print(
        f"LOADED window_candidates: "
        f"{window_candidates.shape}"
    )

else:

    window_candidates = None

    print(
        "NOT FOUND: window candidate metrics."
    )


# ------------------------------------------------------------
# 4. Frozen SPLIT-001 file assignment — OPTIONAL
# ------------------------------------------------------------

FILE_SPLIT_PATH = (
    SPLIT_DIR
    / "file_split_v001.csv"
)

if FILE_SPLIT_PATH.exists():

    file_split = pd.read_csv(
        FILE_SPLIT_PATH
    )

    print(
        f"LOADED file_split: "
        f"{file_split.shape}"
    )

else:

    file_split = None

    print(
        "NOT FOUND: SPLIT-001 file assignment."
    )


# ============================================================
# LOAD REQUIRED FROZEN PROJECT SPECIFICATIONS
# ============================================================

import yaml


# ------------------------------------------------------------
# PREP-001 — REQUIRED
# ------------------------------------------------------------

PREPROCESSING_CONFIG_PATH = (
    CONFIG_DIR
    / "preprocessing_v001.yaml"
)

if not PREPROCESSING_CONFIG_PATH.exists():

    raise FileNotFoundError(
        "Required PREP-001 configuration is missing:\n"
        f"{PREPROCESSING_CONFIG_PATH}"
    )


try:

    with open(
        PREPROCESSING_CONFIG_PATH,
        "r",
        encoding="utf-8",
    ) as file:

        preprocessing_config = (
            yaml.safe_load(file)
        )

except yaml.YAMLError as error:

    raise RuntimeError(
        "PREP-001 exists but is invalid YAML.\n"
        f"{error}"
    )


if (
    preprocessing_config[
        "specification"
    ]["preprocessing_version"]
    != "PREP-001"
):

    raise RuntimeError(
        "Unexpected preprocessing version."
    )


print(
    "LOADED preprocessing_config: PREP-001"
)


# ------------------------------------------------------------
# SPLIT-001 — REQUIRED
# ------------------------------------------------------------

SPLIT_CONFIG_PATH = (
    CONFIG_DIR
    / "split_v001.yaml"
)

if not SPLIT_CONFIG_PATH.exists():

    raise FileNotFoundError(
        "Required SPLIT-001 configuration is missing:\n"
        f"{SPLIT_CONFIG_PATH}"
    )


try:

    with open(
        SPLIT_CONFIG_PATH,
        "r",
        encoding="utf-8",
    ) as file:

        split_config = (
            yaml.safe_load(file)
        )

except yaml.YAMLError as error:

    raise RuntimeError(
        "SPLIT-001 exists but is invalid YAML.\n"
        f"{error}"
    )


if (
    split_config[
        "specification"
    ]["split_version"]
    != "SPLIT-001"
):

    raise RuntimeError(
        "Unexpected split version."
    )


print(
    "LOADED split_config: SPLIT-001"
)


# ------------------------------------------------------------
# FILE SPLIT — REQUIRED
# ------------------------------------------------------------

FILE_SPLIT_PATH = (
    SPLIT_DIR
    / "file_split_v001.csv"
)

if not FILE_SPLIT_PATH.exists():

    raise FileNotFoundError(
        "Required SPLIT-001 assignment file is missing:\n"
        f"{FILE_SPLIT_PATH}"
    )


file_split = pd.read_csv(
    FILE_SPLIT_PATH
)

print(
    f"LOADED file_split: "
    f"{file_split.shape}"
)


# ============================================================
# DERIVE FROZEN RUNTIME CONSTANTS
# ============================================================

# ------------------------------------------------------------
# PREP-001 identity
# ------------------------------------------------------------

PREPROCESSING_VERSION = (
    preprocessing_config[
        "specification"
    ]["preprocessing_version"]
)

DATASET_SUBSET = (
    preprocessing_config[
        "specification"
    ]["dataset_subset"]
)


# ------------------------------------------------------------
# Windowing
# ------------------------------------------------------------

WINDOW_DURATION_S = float(
    preprocessing_config[
        "windowing"
    ]["window_duration_s"]
)

OVERLAP_FRACTION = float(
    preprocessing_config[
        "windowing"
    ]["overlap_fraction"]
)

HOP_DURATION_S = float(
    preprocessing_config[
        "windowing"
    ]["hop_duration_s"]
)


# ------------------------------------------------------------
# Source-data definition
# ------------------------------------------------------------

SOURCE_SAMPLE_RATE_HZ = int(
    preprocessing_config[
        "source_data"
    ]["source_sample_rate_hz"]
)

EXPECTED_CHANNELS = int(
    preprocessing_config[
        "source_data"
    ]["expected_channels"]
)

RPM_SCALE_FACTOR = float(
    preprocessing_config[
        "source_data"
    ]["rpm_scale_factor"]
)

TORQUE_SCALE_FACTOR_NM = float(
    preprocessing_config[
        "source_data"
    ]["torque_scale_factor_nm"]
)


# ------------------------------------------------------------
# YAML channel numbers are human-readable 1-based numbers.
# NumPy arrays use 0-based indexing.
# ------------------------------------------------------------

RPM_CHANNEL_INDEX = (
    int(
        preprocessing_config[
            "source_data"
        ]["annotation_channels"]["rpm"]
    )
    - 1
)

TORQUE_CHANNEL_INDEX = (
    int(
        preprocessing_config[
            "source_data"
        ]["annotation_channels"]["torque"]
    )
    - 1
)


# ------------------------------------------------------------
# Audio processing
# ------------------------------------------------------------

TARGET_SAMPLE_RATE_HZ = int(
    preprocessing_config[
        "audio_processing"
    ]["target_sample_rate_hz"]
)

AUDIO_CHANNEL_STRATEGY = (
    preprocessing_config[
        "audio_processing"
    ]["channel_strategy"]
)


# ------------------------------------------------------------
# Steady-state criterion
# ------------------------------------------------------------

ABSOLUTE_RPM_LIMIT = float(
    preprocessing_config[
        "operating_state"
    ]["steady_state_rule"][
        "maximum_absolute_range_rpm"
    ]
)

RELATIVE_RPM_LIMIT = float(
    preprocessing_config[
        "operating_state"
    ]["steady_state_rule"][
        "maximum_relative_range_fraction"
    ]
)


# ------------------------------------------------------------
# SPLIT-001 identity
# ------------------------------------------------------------

SPLIT_VERSION = (
    split_config[
        "specification"
    ]["split_version"]
)


print("Frozen runtime constants restored.")
print()
print(f"Preprocessing:       {PREPROCESSING_VERSION}")
print(f"Split:               {SPLIT_VERSION}")
print(f"Dataset subset:      {DATASET_SUBSET}")
print(f"Window duration:     {WINDOW_DURATION_S} s")
print(f"Overlap:             {100 * OVERLAP_FRACTION:.0f}%")
print(f"Source sample rate:  {SOURCE_SAMPLE_RATE_HZ} Hz")
print(f"Target sample rate:  {TARGET_SAMPLE_RATE_HZ} Hz")
print(f"Expected channels:   {EXPECTED_CHANNELS}")
print(f"RPM channel index:   {RPM_CHANNEL_INDEX}")
print(f"Torque channel idx:  {TORQUE_CHANNEL_INDEX}")
print(f"Channel strategy:    {AUDIO_CHANNEL_STRATEGY}")

# ============================================================
# VALIDATE FROZEN RUNTIME CONSTANTS
# ============================================================

assert PREPROCESSING_VERSION == "PREP-001", (
    f"Expected PREP-001, found {PREPROCESSING_VERSION}"
)

assert SPLIT_VERSION == "SPLIT-001", (
    f"Expected SPLIT-001, found {SPLIT_VERSION}"
)

assert SOURCE_SAMPLE_RATE_HZ == 48000, (
    "Unexpected PREP-001 source sample rate."
)

assert TARGET_SAMPLE_RATE_HZ == 16000, (
    "Unexpected PREP-001 target sample rate."
)

assert EXPECTED_CHANNELS == 4, (
    "Unexpected PREP-001 channel count."
)

assert WINDOW_DURATION_S > 0

assert 0 <= OVERLAP_FRACTION < 1

assert RPM_CHANNEL_INDEX < EXPECTED_CHANNELS

assert TORQUE_CHANNEL_INDEX < EXPECTED_CHANNELS

assert file_split["file_id"].is_unique

assert set(
    file_split["split"].unique()
) == {
    "train",
    "validation",
    "test",
}

print(
    "PASS: Frozen runtime constants validated."
)

# ============================================================
# LOAD SAMPLE-MANIFEST-001
# ============================================================

SAMPLE_MANIFEST_PATH = (
    DRIVE_MANIFEST_DIR
    / "sample_manifest_v001.csv"
)

if not SAMPLE_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        "Required SAMPLE-MANIFEST-001 "
        "is missing:\n"
        f"{SAMPLE_MANIFEST_PATH}"
    )


sample_manifest = pd.read_csv(
    SAMPLE_MANIFEST_PATH
)


assert (
    sample_manifest[
        "sample_id"
    ].is_unique
)

assert set(
    sample_manifest[
        "preprocessing_version"
    ]
) == {
    "PREP-001"
}

assert set(
    sample_manifest[
        "split_version"
    ]
) == {
    "SPLIT-001"
}


print(
    f"LOADED sample_manifest: "
    f"{sample_manifest.shape}"
)

# ============================================================
# PROJECT SESSION READINESS SUMMARY
# ============================================================

print()
print("=" * 64)
print("NVH DEEP-LEARNING PROJECT SESSION READY")
print("=" * 64)

print(
    f"Raw manifest:       "
    f"{len(raw_manifest):,} source files"
)

print(
    f"Preprocessing:      "
    f"{PREPROCESSING_VERSION}"
)

print(
    f"Dataset split:      "
    f"{SPLIT_VERSION}"
)

print(
    f"Window:             "
    f"{WINDOW_DURATION_S:.1f} s"
)

print(
    f"Overlap:            "
    f"{100 * OVERLAP_FRACTION:.0f}%"
)

print(
    f"Source sample rate: "
    f"{SOURCE_SAMPLE_RATE_HZ:,} Hz"
)

print(
    f"Model sample rate:  "
    f"{TARGET_SAMPLE_RATE_HZ:,} Hz"
)

print(
    f"Audio strategy:     "
    f"{AUDIO_CHANNEL_STRATEGY}"
)

print()
print("SPLIT-001:")

print(
    file_split[
        "split"
    ]
    .value_counts()
    .to_string()
)

if (
    "sample_manifest" in globals()
    and
    sample_manifest is not None
):

    print()
    print(
        f"Sample manifest:    "
        f"{len(sample_manifest):,} samples"
    )

else:

    print()
    print(
        "Sample manifest:    "
        "not generated yet"
    )

print("=" * 64)

print(
    f"Sample manifest:    "
    f"{len(sample_manifest):,} samples"
)

# ============================================================
# LOAD ORDER-001 (added after initiating 06_bseline_feature_generation.ipynb)
# ============================================================

ORDER_CONFIG_PATH = (
    CONFIG_DIR
    / "order_analysis_v001.yaml"
)

if not ORDER_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"ORDER-001 configuration not found:\n"
        f"{ORDER_CONFIG_PATH}"
    )

with open(
    ORDER_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:
    order_analysis_config = yaml.safe_load(file)

ORDER_ANALYSIS_VERSION = (
    order_analysis_config[
        "specification"
    ]["order_analysis_version"]
)

assert ORDER_ANALYSIS_VERSION == "ORDER-001"

print(
    "PASS: ORDER-001 loaded."
)

# ============================================================
# LOAD FEAT-001 — REQUIRED FROM NOTEBOOK 07 ONWARD
# ============================================================

FEATURE_CONFIG_PATH = (
    CONFIG_DIR
    / "features_v001.yaml"
)

if not FEATURE_CONFIG_PATH.exists():
    raise FileNotFoundError(
        "Required FEAT-001 configuration "
        "is missing:\n"
        f"{FEATURE_CONFIG_PATH}"
    )


try:

    with open(
        FEATURE_CONFIG_PATH,
        "r",
        encoding="utf-8",
    ) as file:

        feature_config = yaml.safe_load(
            file
        )

except yaml.YAMLError as error:

    raise RuntimeError(
        "FEAT-001 exists but is invalid YAML.\n"
        f"{error}"
    )


FEATURE_VERSION = (
    feature_config[
        "specification"
    ]["feature_version"]
)

assert FEATURE_VERSION == "FEAT-001", (
    f"Expected FEAT-001, "
    f"found {FEATURE_VERSION}"
)


# ------------------------------------------------------------
# Load persistent FEAT-001 feature table
# ------------------------------------------------------------

BASELINE_FEATURE_PATH = (
    PROJECT_DRIVE
    / "data"
    / "cached_features"
    / "FEAT-001"
    / "baseline_features_v001.parquet"
)

if not BASELINE_FEATURE_PATH.exists():

    raise FileNotFoundError(
        "Required FEAT-001 feature table "
        "is missing:\n"
        f"{BASELINE_FEATURE_PATH}"
    )


baseline_features = pd.read_parquet(
    BASELINE_FEATURE_PATH
)


# ------------------------------------------------------------
# Validate frozen feature table
# ------------------------------------------------------------

assert (
    baseline_features[
        "sample_id"
    ].is_unique
)

assert (
    baseline_features[
        "sample_id"
    ].notna().all()
)

assert set(
    baseline_features[
        "feature_version"
    ].unique()
) == {
    "FEAT-001"
}

assert set(
    baseline_features[
        "split"
    ].unique()
) == {
    "train",
    "validation",
    "test",
}

assert (
    len(baseline_features)
    ==
    len(sample_manifest)
)

assert set(
    baseline_features[
        "sample_id"
    ]
) == set(
    sample_manifest[
        "sample_id"
    ]
)


print(
    f"LOADED baseline_features: "
    f"{baseline_features.shape}"
)

print(
    f"Feature version: "
    f"{FEATURE_VERSION}"
)

print(
    "PASS: FEAT-001 restored and validated."
)

# ------------------------------------------------------------
# Updating Readiness Summary
# ------------------------------------------------------------
print()
print("=" * 68)
print("MODELING ENVIRONMENT READY")
print("=" * 68)

print(
    f"Preprocessing:       "
    f"{PREPROCESSING_VERSION}"
)

print(
    f"Dataset split:       "
    f"{SPLIT_VERSION}"
)

print(
    f"Order analysis:      "
    f"{ORDER_ANALYSIS_VERSION}"
)

print(
    f"Feature definition:  "
    f"{FEATURE_VERSION}"
)

print(
    f"Sample manifest:     "
    f"{len(sample_manifest):,} samples"
)

print(
    f"Feature table:       "
    f"{len(baseline_features):,} samples"
)

print(
    f"Feature columns:     "
    f"{baseline_features.shape[1]}"
)

print("=" * 68)

Standard project libraries imported.
LOADED raw_manifest: (767, 21)
LOADED prep001_targets: (16503, 23)
LOADED window_candidates: (58653, 16)
LOADED file_split: (767, 11)
LOADED preprocessing_config: PREP-001
LOADED split_config: SPLIT-001
LOADED file_split: (767, 11)
Frozen runtime constants restored.

Preprocessing:       PREP-001
Split:               SPLIT-001
Dataset subset:      A_full_set
Window duration:     1.0 s
Overlap:             50%
Source sample rate:  48000 Hz
Target sample rate:  16000 Hz
Expected channels:   4
RPM channel index:   2
Torque channel idx:  3
Channel strategy:    mono_average
PASS: Frozen runtime constants validated.
LOADED sample_manifest: (16503, 34)

NVH DEEP-LEARNING PROJECT SESSION READY
Raw manifest:       767 source files
Preprocessing:      PREP-001
Dataset split:      SPLIT-001
Window:             1.0 s
Overlap:            50%
Source sample rate: 48,000 Hz
Model sample rate:  16,000 Hz
Audio strategy:     mono_average

SPLIT-001:
split
train      

In [6]:
# Step 74 was to Create Notebook 10
# Create a new Colab notebook and save it immediately to GitHub as:
# notebooks/10_multitask_rpm_torque_model.ipynb
# Run your standard Startup Cells 1–5 in Notebook 10.

# Step 75 — Freeze the shared MTL population
# MTL-001 will use exactly the existing TORQUE-SPEC-001 population because
# every sample must have both a valid steady RPM target and
# a valid steady torque target.

# Step 75.1 — Restore and validate the population against the HDF5 cache
# ============================================================
# STEP 75.1 — FREEZE MTL-001 SHARED POPULATION
# ============================================================

import h5py
import numpy as np
import pandas as pd
import yaml

MTL_POPULATION_PATH = (
    PROJECT_DRIVE
    / "data"
    / "manifests"
    / "torque_population_v001.csv"
)


CNN_LOGMEL_CACHE_PATH = (
    PROJECT_DRIVE
    / "data"
    / "cached_features"
    / "CNN-001"
    / "logmel_v001.h5"
)


mtl_population = pd.read_csv(
    MTL_POPULATION_PATH
)


with h5py.File(
    CNN_LOGMEL_CACHE_PATH,
    "r",
) as h5:

    cache_sample_ids = (
        h5["sample_id"]
        .asstr()[:]
    )

    cache_splits = (
        h5["split"]
        .asstr()[:]
    )

    cache_shape = (
        h5["log_mel"].shape
    )


mtl_sample_id_set = set(
    mtl_population[
        "sample_id"
    ]
)


cache_mtl_mask = np.array(
    [
        sample_id
        in mtl_sample_id_set

        for sample_id
        in cache_sample_ids
    ],
    dtype=bool,
)


MTL_TRAIN_INDICES = np.where(
    cache_mtl_mask
    &
    (
        cache_splits
        == "train"
    )
)[0]


MTL_VALIDATION_INDICES = np.where(
    cache_mtl_mask
    &
    (
        cache_splits
        == "validation"
    )
)[0]


MTL_TEST_INDICES = np.where(
    cache_mtl_mask
    &
    (
        cache_splits
        == "test"
    )
)[0]


assert len(MTL_TRAIN_INDICES) == 7739
assert len(MTL_VALIDATION_INDICES) == 1749
assert len(MTL_TEST_INDICES) == 1815


LOG_MEL_SHAPE = (
    int(cache_shape[1]),
    int(cache_shape[2]),
)


print(
    f"MTL train:      "
    f"{len(MTL_TRAIN_INDICES):,}"
)

print(
    f"MTL validation: "
    f"{len(MTL_VALIDATION_INDICES):,}"
)

print(
    f"MTL test:       "
    f"{len(MTL_TEST_INDICES):,}"
)

print(
    f"Log-mel shape:  "
    f"{LOG_MEL_SHAPE}"
)

print(
    "PASS: Shared MTL population frozen."
)

MTL train:      7,739
MTL validation: 1,749
MTL test:       1,815
Log-mel shape:  (64, 59)
PASS: Shared MTL population frozen.


In [7]:
# Step 75.2 — Save the population record

# ============================================================
# STEP 75.2 — SAVE MTL POPULATION RECORD
# ============================================================

MTL_POPULATION_RECORD_PATH = (
    REPO_DIR
    / "data"
    / "mtl001_population_v001_record.yaml"
)


mtl_population_record = {

    "model_version":
        "MTL-001",

    "population_source":
        "TORQUE-SPEC-001",

    "population_manifest":
        (
            "data/manifests/"
            "torque_population_v001.csv"
        ),

    "required_targets": [
        "rpm_mean",
        "torque_mean_nm",
    ],

    "sample_counts": {

        "train":
            int(
                len(
                    MTL_TRAIN_INDICES
                )
            ),

        "validation":
            int(
                len(
                    MTL_VALIDATION_INDICES
                )
            ),

        "test":
            int(
                len(
                    MTL_TEST_INDICES
                )
            ),
    },

    "log_mel_cache":
        "CNN-001/logmel_v001.h5",

    "test_used_for_model_selection":
        False,
}


with open(
    MTL_POPULATION_RECORD_PATH,
    "w",
    encoding="utf-8",
) as file:

    yaml.safe_dump(
        mtl_population_record,
        file,
        sort_keys=False,
    )


print(
    f"Saved:\n"
    f"{MTL_POPULATION_RECORD_PATH}"
)

Saved:
/content/engine-nvh-deep-learning/data/mtl001_population_v001_record.yaml


In [8]:
# Step 76 — Freeze fair validation benchmarks for MTL-001
# For MTL-001, both incumbent models must be measured on the exact
# same 1,749 validation windows.

# Step 76.1 — Evaluate BASE-002 and TBASE-002 on the shared population

# ============================================================
# STEP 76.1 — MTL-001 SHARED-POPULATION BENCHMARKS
# ============================================================

import joblib

BASELINE_FEATURE_PATH = (
    PROJECT_DRIVE
    / "data"
    / "cached_features"
    / "FEAT-001"
    / "baseline_features_v001.parquet"
)


BASE002_MODEL_PATH = (
    PROJECT_DRIVE
    / "models"
    / "baseline"
    / "base002_random_forest_rpm_v001.joblib"
)


TBASE002_MODEL_PATH = (
    PROJECT_DRIVE
    / "models"
    / "torque_baseline"
    / "tbase002_random_forest_torque_v001.joblib"
)


baseline_features = pd.read_parquet(
    BASELINE_FEATURE_PATH
)

base002_model = joblib.load(
    BASE002_MODEL_PATH
)

tbase002_model = joblib.load(
    TBASE002_MODEL_PATH
)


rpm_feature_columns = list(
    base002_model.feature_names_in_
)

torque_feature_columns = list(
    tbase002_model.feature_names_in_
)


assert (
    rpm_feature_columns
    ==
    torque_feature_columns
)


MTL_FEATURE_COLUMNS = (
    rpm_feature_columns
)


mtl_validation_features = (

    mtl_population.loc[
        mtl_population[
            "split"
        ] == "validation",
        [
            "sample_id",
            "rpm_mean",
            "torque_mean_nm",
        ],
    ]

    .merge(

        baseline_features[
            [
                "sample_id",
                *MTL_FEATURE_COLUMNS,
            ]
        ],

        on="sample_id",

        how="inner",

        validate="one_to_one",
    )
)


X_mtl_validation = (
    mtl_validation_features[
        MTL_FEATURE_COLUMNS
    ]
)


rpm_validation_true = (
    mtl_validation_features[
        "rpm_mean"
    ].to_numpy()
)


torque_validation_true = (
    mtl_validation_features[
        "torque_mean_nm"
    ].to_numpy()
)


rpm_baseline_pred = (
    base002_model.predict(
        X_mtl_validation
    )
)


torque_baseline_pred = (
    tbase002_model.predict(
        X_mtl_validation
    )
)

In [9]:
# Calculate and save:
# ============================================================
# STEP 76.2 — SAVE MTL-001 VALIDATION BENCHMARKS
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


def regression_metrics(
    y_true,
    y_pred,
):

    abs_error = np.abs(
        y_pred
        -
        y_true
    )

    return {

        "mae":
            float(
                mean_absolute_error(
                    y_true,
                    y_pred,
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    mean_squared_error(
                        y_true,
                        y_pred,
                    )
                )
            ),

        "r2":
            float(
                r2_score(
                    y_true,
                    y_pred,
                )
            ),

        "p95_absolute_error":
            float(
                np.percentile(
                    abs_error,
                    95,
                )
            ),
    }


rpm_benchmark_metrics = (
    regression_metrics(
        rpm_validation_true,
        rpm_baseline_pred,
    )
)


torque_benchmark_metrics = (
    regression_metrics(
        torque_validation_true,
        torque_baseline_pred,
    )
)


MTL_TABLE_DIR = (
    REPO_DIR
    / "results"
    / "tables"
    / "multitask"
)

MTL_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


mtl_benchmarks = pd.DataFrame(
    [
        {
            "target":
                "rpm_mean",

            "benchmark_id":
                "BASE-002",

            **rpm_benchmark_metrics,
        },

        {
            "target":
                "torque_mean_nm",

            "benchmark_id":
                "TBASE-002",

            **torque_benchmark_metrics,
        },
    ]
)


MTL_BENCHMARK_PATH = (
    MTL_TABLE_DIR
    / "mtl001_validation_benchmarks_v001.csv"
)


mtl_benchmarks.to_csv(
    MTL_BENCHMARK_PATH,
    index=False,
)


display(
    mtl_benchmarks
)

# These become the exact MTL-001 validation hurdles.

,target,benchmark_id,mae,rmse,r2,p95_absolute_error
0,rpm_mean,BASE-002,38.386810,129.145184,0.993772,188.836825
1,torque_mean_nm,TBASE-002,23.321647,41.574599,0.950266,104.451214


In [10]:
# Step 77 — Freeze target normalization
# Only the shared training population is used.

# ============================================================
# STEP 77 — MTL-001 TARGET NORMALIZATION
# ============================================================

mtl_train_population = (
    mtl_population.loc[
        mtl_population[
            "split"
        ] == "train"
    ]
)


MTL_RPM_MEAN = float(
    mtl_train_population[
        "rpm_mean"
    ].mean()
)

MTL_RPM_STD = float(
    mtl_train_population[
        "rpm_mean"
    ].std(
        ddof=0
    )
)


MTL_TORQUE_MEAN = float(
    mtl_train_population[
        "torque_mean_nm"
    ].mean()
)

MTL_TORQUE_STD = float(
    mtl_train_population[
        "torque_mean_nm"
    ].std(
        ddof=0
    )
)


MTL_TARGET_NORMALIZATION_PATH = (
    REPO_DIR
    / "data"
    / "mtl001_target_normalization_v001.yaml"
)


mtl_target_normalization = {

    "model_version":
        "MTL-001",

    "fit_population":
        "shared_training_split_only",

    "sample_count":
        int(
            len(
                mtl_train_population
            )
        ),

    "rpm": {

        "mean":
            MTL_RPM_MEAN,

        "std":
            MTL_RPM_STD,
    },

    "torque_nm": {

        "mean":
            MTL_TORQUE_MEAN,

        "std":
            MTL_TORQUE_STD,
    },

    "validation_statistics_used":
        False,

    "test_statistics_used":
        False,
}


with open(
    MTL_TARGET_NORMALIZATION_PATH,
    "w",
    encoding="utf-8",
) as file:

    yaml.safe_dump(
        mtl_target_normalization,
        file,
        sort_keys=False,
    )


print(
    f"RPM mean/std: "
    f"{MTL_RPM_MEAN:.2f} / "
    f"{MTL_RPM_STD:.2f}"
)

print(
    f"Torque mean/std: "
    f"{MTL_TORQUE_MEAN:.2f} / "
    f"{MTL_TORQUE_STD:.2f}"
)

print(
    "PASS: MTL target normalization frozen."
)

RPM mean/std: 3017.95 / 1724.30
Torque mean/std: 105.23 / 170.51
PASS: MTL target normalization frozen.


In [11]:
# Step 78 — Freeze MTL-001 specification
# MTL-001 will deliberately use the frequency-aware CNN-003 encoder,
# but add two task heads. That makes this a meaningful experiment:
# 'Can joint RPM+torque supervision make the learned acoustic representation
# more useful than single-target CNN training?'

# ============================================================
# STEP 78 — FREEZE MTL-001 SPECIFICATION
# ============================================================

CNN003_CONFIG_PATH = (
    REPO_DIR
    / "configs"
    / "cnn_rpm_v003.yaml"
)


with open(
    CNN003_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:

    cnn003_config = yaml.safe_load(
        file
    )


MTL001_CONFIG_PATH = (
    REPO_DIR
    / "configs"
    / "multitask_rpm_torque_v001.yaml"
)


mtl001_config = {

    "specification": {

        "model_version":
            "MTL-001",

        "status":
            "frozen_pretraining_spec",
    },

    "population": {

        "source":
            "TORQUE-SPEC-001",

        "train_count":
            7739,

        "validation_count":
            1749,

        "test_count":
            1815,
    },

    "input": {

        "representation":
            "CNN-001 log-mel cache",

        "shape":
            list(
                LOG_MEL_SHAPE
            ),

        "input_normalization":
            (
                "cnn001_normalization_v001.yaml "
                "training-only statistics"
            ),

        "runtime_strategy":
            (
                "Mirror HDF5 cache from Drive to "
                "local Colab storage before training"
            ),
    },

    "targets": {

        "rpm":
            "rpm_mean",

        "torque":
            "torque_mean_nm",

        "normalization_record":
            "mtl001_target_normalization_v001.yaml",
    },

    "architecture": {

        "type":
            "shared_frequency_aware_cnn",

        "conv_channels":
            [16, 32, 64],

        "pooling_sequence":
            [
                [1, 2],
                [1, 2],
                [2, 2],
            ],

        "adaptive_pool_output":
            [32, 1],

        "shared_dense_units":
            64,

        "dropout":
            0.20,

        "heads": [
            "rpm",
            "torque",
        ],
    },

    "training": {

        "rpm_loss":
            "MSELoss_standardized",

        "torque_loss":
            "MSELoss_standardized",

        "rpm_loss_weight":
            0.5,

        "torque_loss_weight":
            0.5,

        "optimizer":
            "AdamW",

        "learning_rate":
            0.001,

        "weight_decay":
            0.0001,

        "batch_size":
            64,

        "maximum_epochs":
            25,

        "early_stopping_patience":
            7,

        "random_seed":
            42,
    },

    "selection": {

        "primary_metric":
            "validation_joint_normalized_mae",

        "definition":
            (
                "0.5*(rpm_MAE/train_rpm_std) + "
                "0.5*(torque_MAE/train_torque_std)"
            ),

        "test_approval_rule":
            (
                "MTL-001 must not be worse than the "
                "frozen shared-population benchmark "
                "on either RPM MAE or torque MAE, "
                "and must improve at least one."
            ),
    },

    "test_used_for_tuning":
        False,
}


with open(
    MTL001_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:

    yaml.safe_dump(
        mtl001_config,
        file,
        sort_keys=False,
    )


print(
    f"Saved:\n"
    f"{MTL001_CONFIG_PATH}"
)

print(
    "PASS: MTL-001 specification frozen."
)

Saved:
/content/engine-nvh-deep-learning/configs/multitask_rpm_torque_v001.yaml
PASS: MTL-001 specification frozen.


In [12]:
# Step 79 — Create the multi-task Dataset module

# ============================================================
# STEP 79 — CREATE src/multitask_dataset.py
# ============================================================

import textwrap

MTL_DATASET_MODULE_PATH = (
    REPO_DIR
    / "src"
    / "multitask_dataset.py"
)


mtl_dataset_code = r'''
import h5py
import numpy as np
import torch

from torch.utils.data import Dataset


class H5LogMelMultiTaskDataset(Dataset):

    def __init__(
        self,
        h5_path,
        indices,
        logmel_mean,
        logmel_std,
        rpm_mean,
        rpm_std,
        torque_mean,
        torque_std,
    ):

        self.h5_path = str(h5_path)

        self.indices = np.asarray(
            indices,
            dtype=np.int64,
        )

        self.logmel_mean = float(logmel_mean)
        self.logmel_std = float(logmel_std)

        self.rpm_mean = float(rpm_mean)
        self.rpm_std = float(rpm_std)

        self.torque_mean = float(torque_mean)
        self.torque_std = float(torque_std)

        self._h5 = None


    def _get_h5(self):

        if self._h5 is None:

            self._h5 = h5py.File(
                self.h5_path,
                "r",
            )

        return self._h5


    def __len__(self):

        return len(self.indices)


    def __getitem__(
        self,
        dataset_index,
    ):

        h5 = self._get_h5()

        cache_index = int(
            self.indices[
                dataset_index
            ]
        )


        x = (
            h5["log_mel"][
                cache_index
            ]
            .astype(
                np.float32
            )
        )


        rpm = float(
            h5["rpm_mean"][
                cache_index
            ]
        )

        torque = float(
            h5["torque_mean_nm"][
                cache_index
            ]
        )


        x = (
            x
            - self.logmel_mean
        ) / self.logmel_std


        rpm_standardized = (
            rpm
            - self.rpm_mean
        ) / self.rpm_std


        torque_standardized = (
            torque
            - self.torque_mean
        ) / self.torque_std


        x_tensor = (
            torch.from_numpy(
                x
            )
            .unsqueeze(0)
        )


        rpm_tensor = torch.tensor(
            rpm_standardized,
            dtype=torch.float32,
        )


        torque_tensor = torch.tensor(
            torque_standardized,
            dtype=torch.float32,
        )


        return (
            x_tensor,
            rpm_tensor,
            torque_tensor,
            cache_index,
        )


    def close(self):

        if self._h5 is not None:

            self._h5.close()

            self._h5 = None


    def __del__(self):

        self.close()
'''


MTL_DATASET_MODULE_PATH.write_text(
    textwrap.dedent(
        mtl_dataset_code
    ).lstrip(),
    encoding="utf-8",
)


print(
    f"Created:\n"
    f"{MTL_DATASET_MODULE_PATH}"
)

Created:
/content/engine-nvh-deep-learning/src/multitask_dataset.py


In [13]:
# Step 80 — Create the MTL-001 model

# ============================================================
# STEP 80 — CREATE src/multitask_models.py
# ============================================================

MTL_MODEL_MODULE_PATH = (
    REPO_DIR
    / "src"
    / "multitask_models.py"
)


mtl_model_code = r'''
import torch
import torch.nn as nn


class MultiTaskRPMTorqueCNN(nn.Module):

    def __init__(
        self,
        dropout=0.20,
    ):

        super().__init__()


        self.encoder = nn.Sequential(

            nn.Conv2d(
                1, 16,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=(1, 2)
            ),


            nn.Conv2d(
                16, 32,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=(1, 2)
            ),


            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=(2, 2)
            ),

            nn.AdaptiveAvgPool2d(
                (32, 1)
            ),
        )


        self.shared_projection = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 32,
                64,
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),
        )


        self.rpm_head = nn.Linear(
            64,
            1,
        )


        self.torque_head = nn.Linear(
            64,
            1,
        )


    def forward(
        self,
        x,
    ):

        x = self.encoder(
            x
        )

        shared = self.shared_projection(
            x
        )


        rpm = self.rpm_head(
            shared
        ).squeeze(-1)


        torque = self.torque_head(
            shared
        ).squeeze(-1)


        return rpm, torque
'''


MTL_MODEL_MODULE_PATH.write_text(
    textwrap.dedent(
        mtl_model_code
    ).lstrip(),
    encoding="utf-8",
)


print(
    f"Created:\n"
    f"{MTL_MODEL_MODULE_PATH}"
)

Created:
/content/engine-nvh-deep-learning/src/multitask_models.py


In [14]:
# Step 81 — Smoke-test MTL-001 without training
# This is NOT a training step. It reads one small batch, performs a
# forward pass, calculates losses, and stops.

# Step 81.1 — Restore input normalization
# ============================================================
# STEP 81.1 — RESTORE MTL INPUT NORMALIZATION
# ============================================================

CNN_NORMALIZATION_PATH = (
    REPO_DIR
    / "data"
    / "cnn001_normalization_v001.yaml"
)


with open(
    CNN_NORMALIZATION_PATH,
    "r",
    encoding="utf-8",
) as file:

    cnn_norm = yaml.safe_load(
        file
    )


MTL_LOGMEL_MEAN = float(
    cnn_norm[
        "log_mel"
    ]["mean"]
)

MTL_LOGMEL_STD = float(
    cnn_norm[
        "log_mel"
    ]["std"]
)

In [15]:
# Step 81.2 — Create train/validation Dataset objects only

# ============================================================
# STEP 81.2 — CREATE MTL TRAIN / VALIDATION DATASETS
# NO TEST DATALOADER
# ============================================================

import sys
import importlib

if str(REPO_DIR) not in sys.path:

    sys.path.insert(
        0,
        str(REPO_DIR),
    )


import src.multitask_dataset

importlib.reload(
    src.multitask_dataset
)


from src.multitask_dataset import (
    H5LogMelMultiTaskDataset
)


mtl_train_dataset = (
    H5LogMelMultiTaskDataset(

        CNN_LOGMEL_CACHE_PATH,

        MTL_TRAIN_INDICES,

        MTL_LOGMEL_MEAN,
        MTL_LOGMEL_STD,

        MTL_RPM_MEAN,
        MTL_RPM_STD,

        MTL_TORQUE_MEAN,
        MTL_TORQUE_STD,
    )
)


mtl_validation_dataset = (
    H5LogMelMultiTaskDataset(

        CNN_LOGMEL_CACHE_PATH,

        MTL_VALIDATION_INDICES,

        MTL_LOGMEL_MEAN,
        MTL_LOGMEL_STD,

        MTL_RPM_MEAN,
        MTL_RPM_STD,

        MTL_TORQUE_MEAN,
        MTL_TORQUE_STD,
    )
)


print(
    f"Train:      "
    f"{len(mtl_train_dataset):,}"
)

print(
    f"Validation: "
    f"{len(mtl_validation_dataset):,}"
)

Train:      7,739
Validation: 1,749


In [16]:
# Step 81.3 — Forward-pass smoke test

# ============================================================
# STEP 81.3 — MTL-001 FORWARD-PASS SMOKE TEST
# ============================================================

import torch
import torch.nn as nn

from torch.utils.data import (
    DataLoader
)


import src.multitask_models

importlib.reload(
    src.multitask_models
)


from src.multitask_models import (
    MultiTaskRPMTorqueCNN
)


smoke_loader = DataLoader(

    mtl_train_dataset,

    batch_size=8,

    shuffle=False,

    num_workers=0,
)


X_smoke, rpm_smoke, torque_smoke, _ = (
    next(
        iter(
            smoke_loader
        )
    )
)


smoke_model = (
    MultiTaskRPMTorqueCNN(
        dropout=0.20
    )
)


with torch.no_grad():

    rpm_prediction, torque_prediction = (
        smoke_model(
            X_smoke
        )
    )


mse = nn.MSELoss()


rpm_loss = mse(
    rpm_prediction,
    rpm_smoke,
)


torque_loss = mse(
    torque_prediction,
    torque_smoke,
)


joint_loss = (
    0.5 * rpm_loss
    +
    0.5 * torque_loss
)


assert rpm_prediction.shape == rpm_smoke.shape
assert torque_prediction.shape == torque_smoke.shape

assert torch.isfinite(
    joint_loss
)


MTL_PARAMETER_COUNT = sum(
    parameter.numel()
    for parameter
    in smoke_model.parameters()
)


print(
    f"Input shape:       "
    f"{X_smoke.shape}"
)

print(
    f"RPM output:        "
    f"{rpm_prediction.shape}"
)

print(
    f"Torque output:     "
    f"{torque_prediction.shape}"
)

print(
    f"Parameters:        "
    f"{MTL_PARAMETER_COUNT:,}"
)

print(
    f"Smoke joint loss:  "
    f"{joint_loss.item():.6f}"
)

print()

print(
    "PASS: MTL-001 forward pipeline ready."
)

print(
    "NO TRAINING HAS OCCURRED."
)

Input shape:       torch.Size([8, 1, 64, 59])
RPM output:        torch.Size([8])
Torque output:     torch.Size([8])
Parameters:        154,786
Smoke joint loss:  0.425788

PASS: MTL-001 forward pipeline ready.
NO TRAINING HAS OCCURRED.


In [17]:
# Step 82 — Freeze pre-training readiness and close today

# Step 82.1 — Create readiness record
# ============================================================
# STEP 82.1 — MTL-001 PRETRAINING READINESS RECORD
# ============================================================

MTL_READINESS_PATH = (
    REPO_DIR
    / "data"
    / "mtl001_pretraining_readiness_v001.yaml"
)


mtl_readiness_record = {

    "model_version":
        "MTL-001",

    "specification_frozen":
        True,

    "population_validated":
        True,

    "population_counts": {

        "train":
            7739,

        "validation":
            1749,

        "test":
            1815,
    },

    "validation_benchmarks_frozen":
        True,

    "target_normalization_frozen":
        True,

    "dataset_module_validated":
        True,

    "model_module_validated":
        True,

    "forward_pass_validated":
        True,

    "parameter_count":
        int(
            MTL_PARAMETER_COUNT
        ),

    "training_started":
        False,

    "epochs_completed":
        0,

    "test_loader_created":
        False,

    "test_evaluated":
        False,

    "next_step":
        "Step 83 — MTL-001 training",
}


with open(
    MTL_READINESS_PATH,
    "w",
    encoding="utf-8",
) as file:

    yaml.safe_dump(
        mtl_readiness_record,
        file,
        sort_keys=False,
    )


print(
    f"Saved:\n"
    f"{MTL_READINESS_PATH}"
)

print()

print(
    "PASS: MTL-001 is frozen and ready "
    "for tomorrow's training session."
)

Saved:
/content/engine-nvh-deep-learning/data/mtl001_pretraining_readiness_v001.yaml

PASS: MTL-001 is frozen and ready for tomorrow's training session.


In [18]:
# Step 82.2 — Save Notebook 10
# Save: notebooks/10_multitask_rpm_torque_model.ipynb directly to GitHub.

# Step 82.3 — Stage MTL pre-training artifacts
# ============================================================
# STEP 82.3 — STAGE MTL-001 PRETRAINING ARTIFACTS
# ============================================================

import subprocess


paths_to_add = [
    "configs/multitask_rpm_torque_v001.yaml",
    "src/multitask_dataset.py",
    "src/multitask_models.py",
    "data/mtl001_population_v001_record.yaml",
    "data/mtl001_target_normalization_v001.yaml",
    "data/mtl001_pretraining_readiness_v001.yaml",
    "results/tables/multitask",
]


existing_paths = [
    path
    for path in paths_to_add
    if (
        REPO_DIR
        / path
    ).exists()
]


subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "add",
        *existing_paths,
    ],
    check=True,
)


print(
    subprocess.run(
        [
            "git",
            "-C",
            str(REPO_DIR),
            "status",
            "--short",
        ],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
)

A  configs/multitask_rpm_torque_v001.yaml
A  data/mtl001_population_v001_record.yaml
A  data/mtl001_pretraining_readiness_v001.yaml
A  data/mtl001_target_normalization_v001.yaml
A  results/tables/multitask/mtl001_validation_benchmarks_v001.csv
A  src/multitask_dataset.py
A  src/multitask_models.py



In [19]:
# Step 82.4 — Commit, reconcile and push

# ============================================================
# STEP 82.4 — COMMIT MTL-001 PRETRAINING STATE
# ============================================================

commit_result = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "commit",
        "-m",
        "Freeze MTL-001 pretraining specification",
    ],
    capture_output=True,
    text=True,
)


print(
    commit_result.stdout
)

if commit_result.stderr:
    print(
        commit_result.stderr
    )

[main 74d2014] Freeze MTL-001 pretraining specification
 7 files changed, 371 insertions(+)
 create mode 100644 configs/multitask_rpm_torque_v001.yaml
 create mode 100644 data/mtl001_population_v001_record.yaml
 create mode 100644 data/mtl001_pretraining_readiness_v001.yaml
 create mode 100644 data/mtl001_target_normalization_v001.yaml
 create mode 100644 results/tables/multitask/mtl001_validation_benchmarks_v001.csv
 create mode 100644 src/multitask_dataset.py
 create mode 100644 src/multitask_models.py



In [20]:
# Rebase and Push

# ============================================================
# STEP 82.5 — REBASE AND PUSH
# ============================================================

current_branch = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "branch",
        "--show-current",
    ],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()


pull_result = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "pull",
        "--rebase",
        "origin",
        current_branch,
    ],
    capture_output=True,
    text=True,
)


if pull_result.returncode != 0:

    print(
        pull_result.stdout
    )

    print(
        pull_result.stderr
    )

    raise RuntimeError(
        "git pull --rebase failed. "
        "Do not force push."
    )


subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "push",
        "origin",
        current_branch,
    ],
    check=True,
)


print(
    "PASS: MTL-001 pretraining state pushed."
)

PASS: MTL-001 pretraining state pushed.


In [21]:
# Step 82.6 — Final synchronization check

# ============================================================
# STEP 82.6 — FINAL PRETRAINING SYNCHRONIZATION CHECK
# ============================================================

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "fetch",
        "origin",
    ],
    check=True,
)


local_head = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "rev-parse",
        "HEAD",
    ],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()


remote_head = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "rev-parse",
        f"origin/{current_branch}",
    ],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()


working_tree = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "status",
        "--porcelain",
    ],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()


assert (
    local_head
    == remote_head
)


print(
    "Local :",
    local_head,
)

print(
    "Remote:",
    remote_head,
)


if working_tree:

    print(
        "\nWARNING: Working tree not clean:"
    )

    print(
        working_tree
    )

else:

    print(
        "\nPASS: Working tree clean."
    )


print()
print("=" * 70)

print(
    "MTL-001 PRETRAINING SESSION CLOSED SAFELY"
)

print(
    "Training started: NO"
)

print(
    "Completed epochs: 0"
)

print(
    "Test evaluated: NO"
)

print(
    "Next step: STEP 83 — MTL-001 TRAINING"
)

print("=" * 70)

Local : 1fd3a6f0e663fe7f6578fc93e94474399fc5f46c
Remote: 1fd3a6f0e663fe7f6578fc93e94474399fc5f46c

PASS: Working tree clean.

MTL-001 PRETRAINING SESSION CLOSED SAFELY
Training started: NO
Completed epochs: 0
Test evaluated: NO
Next step: STEP 83 — MTL-001 TRAINING
